---
title: Indicators
execute:
  enabled: true
---

`q.indicator` contains deterministic market measurements for one instrument at a time. Indicators preserve pandas indexes, never group by `symbol` implicitly, and do not express investment intent.

Use indicators directly in research, dashboards, alerts, and rules, or wrap them in `q.feature.Feature` when the result must be named, versioned, computed per entity, and materialized for model training.

```python
q.indicator.sma(prices["close"], 20)
q.indicator.ema(prices["close"], 20)
q.indicator.relative_strength(prices["close"], benchmark, lookback=21)
```

Native functions are flat under `q.indicator`; provider-specific formulas remain explicit under `q.indicator.talib` and `q.indicator.pandas_ta`. The examples below are **executed live** by Quarto every time the docs are built.

## Sample data

We use qrt's bundled AAPL sample dataset — loaded offline via `q.data.datasets.load`,
no network dependency (see the [Data](../data/index.qmd) tutorial for more on
`q.data`):

In [1]:
import pandas as pd

import qrt as q

aapl = q.data.datasets.load("aapl")
aapl.tail()

,open,high,low,close,volume
datetime,,,,,
2026-07-13,317.019989,323.450012,315.779999,317.309998,43257800
2026-07-14,313.760010,316.190002,311.910004,314.859985,36336800
2026-07-15,317.619995,328.730011,317.320007,327.500000,60957600
2026-07-16,328.010010,334.679993,326.790009,333.260010,62970600
2026-07-17,331.980011,334.989990,329.000000,333.739990,63365300


Calculate reusable market measurements directly. These values are not registered or persisted as model features until they are wrapped by `q.feature.Feature`:

In [2]:
sma_20 = q.indicator.sma(aapl["close"], 20)
sma_100 = q.indicator.sma(aapl["close"], 100)

price_indicators = pd.DataFrame({"close": aapl["close"], "sma_20": sma_20, "sma_100": sma_100})
price_indicators.tail()

,close,sma_20,sma_100
datetime,,,
2026-07-13,317.309998,299.187001,279.561089
2026-07-14,314.859985,300.373500,280.068622
2026-07-15,327.500000,301.927499,280.740221
2026-07-16,333.260010,303.628500,281.429456
2026-07-17,333.739990,305.517999,282.107506


In [3]:
q.indicator.ema(aapl["close"], 20).tail()

datetime
2026-07-13    303.524259
2026-07-14    304.603852
2026-07-15    306.784438
2026-07-16    309.305921
2026-07-17    311.632975
Name: close, dtype: float64

To add a native indicator, create `qrt/indicator/_<indicator>.py` with one public function and re-export it from `qrt/indicator/__init__.py`. Keep one canonical formula per native name; provider variants remain under their provider namespace.

## Visualizing indicators

`q.plot.col` renders any indicator DataFrame as an interactive Plotly chart with hover, zoom, and range-selector buttons. Visualization does not register the values as stored features:

In [4]:
fig = q.plot.col(
    price_indicators,
    title="AAPL close price with SMA overlays",
    ylabel="Price",
    height=450,
)
fig.show()